# Evidence calibrated VideoQA workflow
Run one stage at a time. The code is a research starting implementation; no new accuracy result has been produced. Use a clean CUDA-enabled environment, and save a copy of this notebook with outputs. Keep the benchmark test set sealed during development.

The local dataset paths below are based on your earlier thesis setup; verify them. Scripts run from the code folder. The synthetic pilot produces descriptive QA only. A person must review audit examples against the videos before selector fitting.

In [ ]:
from pathlib import Path
import sys, subprocess, json
import torch
print('Python:',sys.version)
print('PyTorch:',torch.__version__,'CUDA:',torch.cuda.is_available())
# Run this notebook from its containing code folder.
CODE = Path.cwd()
assert (CODE/'train.py').exists(), 'Open Jupyter in the extracted code directory'
def run(script, *args):
    subprocess.run([sys.executable, str(CODE/script), *map(str,args)], check=True)
DATA = CODE/'data'
CACHE = CODE/'cache'
VIDEOS = Path(r'F:\MSC\thesis\dataset2\MSVD\Videos\MSVD_Videos\YouTubeClips')
ANNOTATIONS = Path(r'F:\MSC\thesis\dataset2\MSVD\Annotations\MSVD-QA\MSVD-QA')
MAPPING = ANNOTATIONS/'youtube_mapping.txt'  # Set the actual mapping-file location.
assert VIDEOS.exists() and ANNOTATIONS.exists() and MAPPING.exists()


## 1 Prepare manifests
Missing videos and overlapping official video IDs stop preparation. Inspect the report. Add original-source grouping and near-duplicate checks before final experiments.

In [ ]:
run('data.py','--videos',VIDEOS,'--annotations',ANNOTATIONS,'--mapping',MAPPING,'--out',DATA)
print((DATA/'manifest_report.json').read_text())

## 2 Cache features
For a first smoke run, use a small COPY of train/validation manifests. Full extraction below processes the permitted split videos. Feature caching does not train encoders. Never use test features for training or selection.

In [ ]:
for split in ['train','val']:
    run('features.py','--manifest',DATA/f'{split}.jsonl','--cache',CACHE)

## 3 Train supervised scorers and baseline
Start with 20 updates on a tiny copied development set, then use fixed full-data budgets. These 3,000-update settings are starting values, not tuned results. The baseline best checkpoint supplies shared student initialization. The question-only checkpoint is a different trained model.

In [ ]:
run('train.py','--train',DATA/'train.jsonl','--val',DATA/'val.jsonl','--cache',CACHE,'--vocab',DATA/'vocab.json','--output',CODE/'runs/full','--steps',3000,'--seed',42)
run('train.py','--train',DATA/'train.jsonl','--val',DATA/'val.jsonl','--cache',CACHE,'--vocab',DATA/'vocab.json','--output',CODE/'runs/blind','--question-only','--steps',3000,'--seed',42)

## 4 Generate a small descriptive pilot
BLIP runs separately from FLAN-T5 to limit peak memory. Inspect captions and rejected outputs. Do not treat generated answers as ground truth. A caption about a glass cannot establish that the liquid is water. Increase --limit only after reviewing yield and evidence quality.

In [ ]:
run('generate.py','--mode','captions','--manifest',DATA/'train.jsonl','--output',DATA/'captions.jsonl','--limit',200)
run('generate.py','--mode','qa','--manifest',DATA/'train.jsonl','--captions',DATA/'captions.jsonl','--vocab',DATA/'vocab.json','--output',DATA/'candidates.jsonl')

## 5 Cache candidate questions and score them
The current implementation uses three fixed random training-video substitutions, not content-matched hard negatives. Save their IDs. The main-paper hard-negative extension needs a source/content grouping manifest.

In [ ]:
run('features.py','--manifest',DATA/'candidates.jsonl','--cache',CACHE)
run('score.py','--candidates',DATA/'candidates.jsonl','--train-manifest',DATA/'train.jsonl','--cache',CACHE,'--full',CODE/'runs/full/best.pt','--blind',CODE/'runs/blind/best.pt','--output',DATA/'scores.jsonl')

## 6 Export the audit
The CSV partitions candidates by video into fit/development/evaluation groups. Original-source groups must also be checked. Review the annotation packet using the actual clips, record richer rubric labels separately, and fill valid as 0 or 1 only after adjudication. A small pilot may not yield 400 usable candidates; expand the candidate pool or report a smaller audit with uncertainty.

In [ ]:
run('select_data.py','--scores',DATA/'scores.jsonl','--audit',DATA/'audit.csv','--make-audit','--n',400)

## 7 Fit calibration and create matched subsets
This cell should fail while human labels are blank. It removes every audited video from the synthetic training pool. --per-video imposes the stronger per-video count control. Sparse strata reduce the intended retention precision; inspect actual_fraction in the report.

In [ ]:
run('select_data.py','--scores',DATA/'scores.jsonl','--audit',DATA/'audit.csv','--counts',DATA/'answer_counts.json','--output',CODE/'selection','--fraction',0.5,'--per-video')

## 8 Synthetic adaptation followed by identical human fine tuning
All arms share baseline initialization. The synthetic stage uses exactly 1,000 updates; the human stage uses 3,000. Use last.pt for the synthetic-to-human transition so the synthetic budget is fixed rather than chosen from its best intermediate validation score. Final human checkpoints use the same validation rule. Repeat essential experiments with seeds 42, 43, 44 after freezing the recipe.

In [ ]:
for arm in ['random','confidence','selected']:
    run('train.py','--train',CODE/f'selection/{arm}.jsonl','--val',DATA/'val.jsonl','--cache',CACHE,'--vocab',DATA/'vocab.json','--init',CODE/'runs/full/best.pt','--output',CODE/f'runs/{arm}_synthetic','--steps',1000,'--seed',42)
    run('train.py','--train',DATA/'train.jsonl','--val',DATA/'val.jsonl','--cache',CACHE,'--vocab',DATA/'vocab.json','--init',CODE/f'runs/{arm}_synthetic/last.pt','--output',CODE/f'runs/{arm}_final','--steps',3000,'--seed',42)

## 9 Matched total update human control
This arm repeats human QA for 4,000 updates from the same initialization. For an exact two-stage optimizer-reset control, run 1,000 then 3,000 updates as below. All auxiliary arms therefore have the same number of post-initialization optimizer steps.

In [ ]:
run('train.py','--train',DATA/'train.jsonl','--val',DATA/'val.jsonl','--cache',CACHE,'--vocab',DATA/'vocab.json','--init',CODE/'runs/full/best.pt','--output',CODE/'runs/human_extra','--steps',1000,'--seed',42)
run('train.py','--train',DATA/'train.jsonl','--val',DATA/'val.jsonl','--cache',CACHE,'--vocab',DATA/'vocab.json','--init',CODE/'runs/human_extra/last.pt','--output',CODE/'runs/human_final','--steps',3000,'--seed',42)

## 10 Development diagnostics
These are validation diagnostics. Report question-only performance separately from zero-video input. Repeat wrong-video and frame-order tests with several seeds. A bigger drop alone is not evidence of better grounding.

In [ ]:
for arm in ['random','confidence','selected','human']:
    for mode in ['full','zero','wrong','order','single']:
        run('train.py','--val',DATA/'val.jsonl','--cache',CACHE,'--vocab',DATA/'vocab.json','--eval-checkpoint',CODE/f'runs/{arm}_final/best.pt','--mode',mode,'--output',CODE/f'runs/{arm}_{mode}_val.jsonl')
run('bootstrap.py','--a',CODE/'runs/selected_full_val.jsonl','--b',CODE/'runs/random_full_val.jsonl')

## 11 Locked final test
Run only after all prompts, thresholds, training budgets, models, and hyperparameters are frozen. Inspect the boolean gate deliberately. Do not repeatedly tune against these outputs. Test features are computed here for evaluation only.

In [ ]:
FINAL_PROTOCOL_FROZEN = False
if not FINAL_PROTOCOL_FROZEN:
    raise RuntimeError('Freeze the experimental protocol before final test evaluation')
run('features.py','--manifest',DATA/'test.jsonl','--cache',CACHE)
for arm in ['random','confidence','selected','human']:
    run('train.py','--val',DATA/'test.jsonl','--cache',CACHE,'--vocab',DATA/'vocab.json','--eval-checkpoint',CODE/f'runs/{arm}_final/best.pt','--mode','full','--output',CODE/f'runs/{arm}_full_test.jsonl')
run('bootstrap.py','--a',CODE/'runs/selected_full_test.jsonl','--b',CODE/'runs/random_full_test.jsonl')

## 12 Archive the environment and interpret results
Save all manifests, checkpoints, predictions, selected IDs, audit labels and adjudication, and version logs. Fill the manuscript tables only with actual outputs. Calibration-evaluation data must remain independent of fitting and model selection. Multiple seeds and a second dataset are still needed for the stronger paper claims.

In [ ]:
with (CODE/'tested_environment.txt').open('w',encoding='utf-8') as f:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=f,check=True)